# Notebook 30 — Policy-Based Opponent Framework

## The Pokémon Company — PTCG AI Battle Challenge

### Team Jesus

Notebook 29 showed that the current self-play dataset has low gameplay diversity:

- 100 games
- 350 replay steps
- 7 unique states
- 2 unique moves
- 1 winner
- repeated state patterns

Notebook 30 introduces a reusable policy-based multi-agent framework.

Instead of hard-coding every behavior into a different agent class, the framework separates:

- the agent,
- the decision policy,
- the configuration,
- and the final move decision.

## Architecture


Agent
  ↓
Policy
  ↓
State + Legal Moves
  ↓
AgentDecision


## Objectives

1. Define a reusable AgentDecision.
2. Define agent and policy configuration objects.
3. Create a common policy interface.
4. Implement random, greedy, aggressive, defensive, and heuristic policies.
5. Create a configurable policy-driven agent.
6. Build policy and agent factories.
7. Validate all policies against sample game states.
8. Export reusable modules to src/agents.
9. Prepare mixed-opponent self-play for Notebook 31.

# Cell 2 - Imports

In [1]:
from __future__ import annotations

import inspect
import random
import sys

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Mapping, Sequence

print("Python:", sys.version)
print("Notebook 30 initialized.")

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Notebook 30 initialized.


# Cell 3 — Locate the Project

In [2]:
def find_project_root(
    start: Path | None = None,
) -> Path:
    current = (start or Path.cwd()).resolve()

    markers = {
        "notebooks",
        "scripts",
        "src",
        "reports",
    }

    for candidate in [
        current,
        *current.parents,
    ]:
        if all(
            (candidate / marker).exists()
            for marker in markers
        ):
            return candidate

    raise FileNotFoundError(
        "Project root not found."
    )


PROJECT_ROOT = find_project_root()

SRC_DIR = PROJECT_ROOT / "src"
AGENTS_DIR = SRC_DIR / "agents"
REPORT_DIR = PROJECT_ROOT / "reports" / "notebook30"

AGENTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root:", PROJECT_ROOT)
print("Agents directory:", AGENTS_DIR)
print("Report directory:", REPORT_DIR)

assert SRC_DIR.exists()
assert AGENTS_DIR.exists()
assert REPORT_DIR.exists()

Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Agents directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\agents
Report directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook30


# Cell 4 — Agent Decision

In [3]:
@dataclass(frozen=True, slots=True)
class AgentDecision:
    move: Any
    score: float = 0.0
    reason: str = ""
    metadata: dict[str, Any] = field(
        default_factory=dict
    )


sample_decision = AgentDecision(
    move="Evolution Burst",
    score=633.0,
    reason="Highest evaluated move",
    metadata={
        "search_depth": 6,
    },
)

print(sample_decision)

assert sample_decision.move == "Evolution Burst"
assert sample_decision.score == 633.0
assert sample_decision.metadata["search_depth"] == 6

print("AgentDecision validated.")

AgentDecision(move='Evolution Burst', score=633.0, reason='Highest evaluated move', metadata={'search_depth': 6})
AgentDecision validated.


# Cell 5 — Policy Configuration

In [4]:
@dataclass(frozen=True, slots=True)
class PolicyConfig:
    name: str
    policy_type: str

    seed: int | None = None

    aggression_weight: float = 1.0
    defense_weight: float = 1.0
    immediate_value_weight: float = 1.0
    randomness: float = 0.0

    search_depth: int = 0

    metadata: Mapping[str, Any] = field(
        default_factory=dict
    )


random_policy_config = PolicyConfig(
    name="RandomPolicy",
    policy_type="random",
    seed=42,
    randomness=1.0,
)

print(random_policy_config)

assert random_policy_config.policy_type == "random"
assert 0.0 <= random_policy_config.randomness <= 1.0

print("PolicyConfig validated.")

PolicyConfig(name='RandomPolicy', policy_type='random', seed=42, aggression_weight=1.0, defense_weight=1.0, immediate_value_weight=1.0, randomness=1.0, search_depth=0, metadata={})
PolicyConfig validated.


# Cell 6 — Base Policy Interface

In [5]:
class BasePolicy(ABC):

    def __init__(
        self,
        config: PolicyConfig,
    ) -> None:
        self.config = config
        self.name = config.name
        self.policy_type = config.policy_type

        self._rng = random.Random(
            config.seed
        )

    @abstractmethod
    def choose_move(
        self,
        state: Mapping[str, Any],
        legal_moves: Sequence[Any],
    ) -> AgentDecision:
        """
        Choose one move from legal_moves.
        """

    def validate_legal_moves(
        self,
        legal_moves: Sequence[Any],
    ) -> None:
        if not legal_moves:
            raise ValueError(
                f"{self.name} received no legal moves."
            )

    def __repr__(self) -> str:
        return (
            f"{self.__class__.__name__}("
            f"name={self.name!r}, "
            f"policy_type={self.policy_type!r})"
        )


#  Cell 7 — Validate the Policy Interface

In [6]:
print(
    "BasePolicy is abstract:",
    inspect.isabstract(BasePolicy),
)

print(
    "Abstract methods:",
    BasePolicy.__abstractmethods__,
)

assert inspect.isabstract(BasePolicy)
assert "choose_move" in BasePolicy.__abstractmethods__

print("BasePolicy interface validated.")

BasePolicy is abstract: True
Abstract methods: frozenset({'choose_move'})
BasePolicy interface validated.


# Cell 8 — RandomPolicy

In [7]:
class RandomPolicy(BasePolicy):

    def choose_move(
        self,
        state: Mapping[str, Any],
        legal_moves: Sequence[Any],
    ) -> AgentDecision:

        self.validate_legal_moves(
            legal_moves
        )

        move = self._rng.choice(
            list(legal_moves)
        )

        return AgentDecision(
            move=move,
            score=0.0,
            reason="Random legal move",
            metadata={
                "policy": self.name,
            },
        )

# Cell 9 — Validate RandomPolicy

In [8]:
random_policy = RandomPolicy(
    random_policy_config
)

legal_moves = [
    "Evolution Burst",
    "Quick Attack",
    "Retreat",
]

decision = random_policy.choose_move(
    state={},
    legal_moves=legal_moves,
)

print(decision)

assert decision.move in legal_moves
assert decision.reason == "Random legal move"

print()
print("RandomPolicy validated.")

AgentDecision(move='Retreat', score=0.0, reason='Random legal move', metadata={'policy': 'RandomPolicy'})

RandomPolicy validated.


# Cell 10 — GreedyPolicy

In [9]:
class GreedyPolicy(BasePolicy):

    def choose_move(
        self,
        state: Mapping[str, Any],
        legal_moves: Sequence[Any],
    ) -> AgentDecision:

        self.validate_legal_moves(
            legal_moves
        )

        scores = state.get(
            "move_scores",
            {},
        )

        best_move = max(
            legal_moves,
            key=lambda move: scores.get(
                move,
                0.0,
            ),
        )

        return AgentDecision(
            move=best_move,
            score=scores.get(
                best_move,
                0.0,
            ),
            reason="Highest immediate value",
            metadata={
                "policy": self.name,
            },
        )

# Cell 11 — Validate GreedyPolicy 
### GreedyPolicy → immediate value.

In [10]:
greedy_config = PolicyConfig(
    name="GreedyPolicy",
    policy_type="greedy",
)

greedy_policy = GreedyPolicy(
    greedy_config
)

state = {
    "move_scores": {
        "Evolution Burst": 633,
        "Quick Attack": 200,
        "Retreat": -50,
    }
}

decision = greedy_policy.choose_move(
    state=state,
    legal_moves=[
        "Evolution Burst",
        "Quick Attack",
        "Retreat",
    ],
)

print(decision)

assert decision.move == "Evolution Burst"
assert decision.score == 633

print()
print("GreedyPolicy validated.")

AgentDecision(move='Evolution Burst', score=633, reason='Highest immediate value', metadata={'policy': 'GreedyPolicy'})

GreedyPolicy validated.


# Cell 12 — HeuristicPolicy

#### This policy evaluates moves using a weighted score from the game state.

In [11]:
class HeuristicPolicy(BasePolicy):

    def choose_move(
        self,
        state: Mapping[str, Any],
        legal_moves: Sequence[Any],
    ) -> AgentDecision:

        self.validate_legal_moves(
            legal_moves
        )

        heuristic_scores = state.get(
            "heuristic_scores",
            {},
        )

        best_move = max(
            legal_moves,
            key=lambda move: heuristic_scores.get(
                move,
                float("-inf"),
            ),
        )

        return AgentDecision(
            move=best_move,
            score=heuristic_scores.get(
                best_move,
                0.0,
            ),
            reason="Highest heuristic evaluation",
            metadata={
                "policy": self.name,
            },
        )

# Cell 13 — Validate HeuristicPolicy

#### weighted evaluation

In [12]:
heuristic_config = PolicyConfig(
    name="HeuristicPolicy",
    policy_type="heuristic",
)

heuristic_policy = HeuristicPolicy(
    heuristic_config
)

state = {
    "heuristic_scores": {
        "Evolution Burst": 612,
        "Quick Attack": 481,
        "Retreat": 105,
    }
}

decision = heuristic_policy.choose_move(
    state=state,
    legal_moves=[
        "Evolution Burst",
        "Quick Attack",
        "Retreat",
    ],
)

print(decision)

assert decision.move == "Evolution Burst"
assert decision.score == 612

print()
print("HeuristicPolicy validated.")

AgentDecision(move='Evolution Burst', score=612, reason='Highest heuristic evaluation', metadata={'policy': 'HeuristicPolicy'})

HeuristicPolicy validated.


# Cell 14 — SearchPolicy

#### integrates with the search engine that was built in earlier notebooks

In [13]:
class SearchPolicy(BasePolicy):

    def choose_move(
        self,
        state: Mapping[str, Any],
        legal_moves: Sequence[Any],
    ) -> AgentDecision:

        self.validate_legal_moves(
            legal_moves
        )

        search_scores = state.get(
            "search_scores",
            {},
        )

        best_move = max(
            legal_moves,
            key=lambda move: search_scores.get(
                move,
                float("-inf"),
            ),
        )

        return AgentDecision(
            move=best_move,
            score=search_scores.get(
                best_move,
                0.0,
            ),
            reason="Search engine recommendation",
            metadata={
                "policy": self.name,
                "search_depth": self.config.search_depth,
            },
        )

# Cell 15 — Validate SearchPolicy

In [14]:
search_config = PolicyConfig(
    name="SearchPolicy",
    policy_type="search",
    search_depth=6,
)

search_policy = SearchPolicy(
    search_config
)

state = {
    "search_scores": {
        "Evolution Burst": 633,
        "Quick Attack": 611,
        "Retreat": -45,
    }
}

decision = search_policy.choose_move(
    state=state,
    legal_moves=[
        "Evolution Burst",
        "Quick Attack",
        "Retreat",
    ],
)

print(decision)

assert decision.move == "Evolution Burst"
assert decision.metadata["search_depth"] == 6

print()
print("SearchPolicy validated.")

AgentDecision(move='Evolution Burst', score=633, reason='Search engine recommendation', metadata={'policy': 'SearchPolicy', 'search_depth': 6})

SearchPolicy validated.


# Cell 16 — BaseAgent

### Now we'll introduce the Agent itself. The agent doesn't decide moves directly, it delegates to a policy.

In [15]:
@dataclass(slots=True)
class BaseAgent:
    name: str
    policy: BasePolicy

    metadata: dict[str, Any] = field(
        default_factory=dict
    )

    def choose_move(
        self,
        state: Mapping[str, Any],
        legal_moves: Sequence[Any],
    ) -> AgentDecision:

        return self.policy.choose_move(
            state=state,
            legal_moves=legal_moves,
        )

    def __repr__(self) -> str:
        return (
            f"{self.__class__.__name__}("
            f"name={self.name!r}, "
            f"policy={self.policy.name!r})"
        )

# Cell 17 — Validate BaseAgent

In [16]:
agent = BaseAgent(
    name="Random Agent",
    policy=random_policy,
)

decision = agent.choose_move(
    state={},
    legal_moves=[
        "Move A",
        "Move B",
        "Move C",
    ],
)

print(agent)
print()
print(decision)

assert decision.move in [
    "Move A",
    "Move B",
    "Move C",
]

print()
print("BaseAgent validated.")

BaseAgent(name='Random Agent', policy='RandomPolicy')

AgentDecision(move='Move A', score=0.0, reason='Random legal move', metadata={'policy': 'RandomPolicy'})

BaseAgent validated.


# Cell 18 — PolicyFactory

#### Instead of manually creating policies everywhere, we'll centralize construction.

In [17]:
class PolicyFactory:

    _registry = {
        "random": RandomPolicy,
        "greedy": GreedyPolicy,
        "heuristic": HeuristicPolicy,
        "search": SearchPolicy,
    }

    @classmethod
    def create(
        cls,
        config: PolicyConfig,
    ) -> BasePolicy:

        if config.policy_type not in cls._registry:
            raise ValueError(
                f"Unknown policy "
                f"{config.policy_type!r}"
            )

        return cls._registry[
            config.policy_type
        ](config)

    @classmethod
    def available_policies(cls):

        return sorted(
            cls._registry.keys()
        )

# Cell 19 — Validate PolicyFactory

In [18]:
print(
    "Available policies:"
)

print(
    PolicyFactory.available_policies()
)

policy = PolicyFactory.create(
    PolicyConfig(
        name="Factory Greedy",
        policy_type="greedy",
    )
)

print()
print(policy)

assert isinstance(
    policy,
    GreedyPolicy,
)

print()
print("PolicyFactory validated.")

Available policies:
['greedy', 'heuristic', 'random', 'search']

GreedyPolicy(name='Factory Greedy', policy_type='greedy')

PolicyFactory validated.


# Cell 20 — AgentFactory

In [19]:
class AgentFactory:

    @staticmethod
    def create(
        name: str,
        policy_type: str,
        **kwargs,
    ) -> BaseAgent:

        config = PolicyConfig(
            name=f"{policy_type.title()}Policy",
            policy_type=policy_type,
            **kwargs,
        )

        policy = PolicyFactory.create(
            config
        )

        return BaseAgent(
            name=name,
            policy=policy,
        )

# Cell 21 — Validate AgentFactory

In [20]:
agent = AgentFactory.create(
    name="Search Agent",
    policy_type="search",
    search_depth=6,
)

print(agent)

decision = agent.choose_move(
    state={
        "search_scores": {
            "Attack": 50,
            "Retreat": 10,
        }
    },
    legal_moves=[
        "Attack",
        "Retreat",
    ],
)

print()
print(decision)

assert isinstance(
    agent.policy,
    SearchPolicy,
)

assert decision.move == "Attack"

print()
print("AgentFactory validated.")

BaseAgent(name='Search Agent', policy='SearchPolicy')

AgentDecision(move='Attack', score=50, reason='Search engine recommendation', metadata={'policy': 'SearchPolicy', 'search_depth': 6})

AgentFactory validated.


# Cell 22 — Register Policies

In [25]:
import pandas as pd


policy_registry = {
    "random": {
        "purpose": "Baseline exploration",
        "uses_search": False,
    },
    "greedy": {
        "purpose": "Highest immediate value",
        "uses_search": False,
    },
    "heuristic": {
        "purpose": "Weighted board evaluation",
        "uses_search": False,
    },
    "search": {
        "purpose": "Search engine evaluation",
        "uses_search": True,
    },
}

policy_df = (
    pd.DataFrame(policy_registry)
    .T
    .reset_index()
    .rename(
        columns={
            "index": "policy_type",
        }
    )
)

display(policy_df)

assert len(policy_df) == 4
assert set(policy_df["policy_type"]) == {
    "random",
    "greedy",
    "heuristic",
    "search",
}

print()
print("Policy registry validated.")

,policy_type,purpose,uses_search
0,random,Baseline exploration,False
1,greedy,Highest immediate value,False
2,heuristic,Weighted board evaluation,False
3,search,Search engine evaluation,True



Policy registry validated.


# Cell 23 — Architecture Summary

In [26]:
architecture_summary = {
    "notebook": 30,
    "policy_count": int(len(policy_df)),
    "agent_class_count": 1,
    "factory_pattern": True,
    "search_policy_available": True,
    "ready_for_mixed_opponents": True,
    "ready_for_self_play_integration": True,
}

for key, value in architecture_summary.items():
    print(f"{key:32}: {value}")

assert architecture_summary["ready_for_mixed_opponents"]
assert architecture_summary["ready_for_self_play_integration"]

print()
print("Architecture summary validated.")

notebook                        : 30
policy_count                    : 4
agent_class_count               : 1
factory_pattern                 : True
search_policy_available         : True
ready_for_mixed_opponents       : True
ready_for_self_play_integration : True

Architecture summary validated.


# Cell 24 — Export Notebook 30 Reports

In [27]:
import json


policy_file = (
    REPORT_DIR
    / "policy_registry.csv"
)

summary_file = (
    REPORT_DIR
    / "architecture_summary.json"
)

policy_df.to_csv(
    policy_file,
    index=False,
)

with open(
    summary_file,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        architecture_summary,
        file,
        indent=2,
    )

print("Exported:")
print(policy_file)
print(summary_file)

Exported:
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook30\policy_registry.csv
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook30\architecture_summary.json


# Cell 25 — Validate Reports

In [28]:
expected_report_files = [
    policy_file,
    summary_file,
]

missing_report_files = [
    path
    for path in expected_report_files
    if not path.exists()
]

assert not missing_report_files, missing_report_files

for path in expected_report_files:
    print(
        f"[OK] {path.name:<32} "
        f"{path.stat().st_size:>8,} bytes"
    )

print()
print("Notebook 30 reports validated.")


[OK] policy_registry.csv                   187 bytes
[OK] architecture_summary.json             217 bytes

Notebook 30 reports validated.


# Cell 26 — Export decisions.py

In [29]:
decisions_module = '''from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any


@dataclass(frozen=True, slots=True)
class AgentDecision:
    move: Any
    score: float = 0.0
    reason: str = ""
    metadata: dict[str, Any] = field(
        default_factory=dict
    )
'''

decisions_file = AGENTS_DIR / "decisions.py"

decisions_file.write_text(
    decisions_module,
    encoding="utf-8",
)

print("[OK]", decisions_file)

[OK] D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\agents\decisions.py


# Cell 27 — Export configs.py

In [30]:
configs_module = '''from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Mapping


@dataclass(frozen=True, slots=True)
class PolicyConfig:
    name: str
    policy_type: str

    seed: int | None = None

    aggression_weight: float = 1.0
    defense_weight: float = 1.0
    immediate_value_weight: float = 1.0
    randomness: float = 0.0

    search_depth: int = 0

    metadata: Mapping[str, Any] = field(
        default_factory=dict
    )
'''

configs_file = AGENTS_DIR / "configs.py"

configs_file.write_text(
    configs_module,
    encoding="utf-8",
)

print("[OK]", configs_file)

[OK] D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\agents\configs.py


# Cell 28 — Export policies.py

In [31]:
policies_module = '''from __future__ import annotations

import random

from abc import ABC, abstractmethod
from typing import Any, Mapping, Sequence

from .configs import PolicyConfig
from .decisions import AgentDecision


class BasePolicy(ABC):

    def __init__(
        self,
        config: PolicyConfig,
    ) -> None:
        self.config = config
        self.name = config.name
        self.policy_type = config.policy_type
        self._rng = random.Random(config.seed)

    @abstractmethod
    def choose_move(
        self,
        state: Mapping[str, Any],
        legal_moves: Sequence[Any],
    ) -> AgentDecision:
        ...

    def validate_legal_moves(
        self,
        legal_moves: Sequence[Any],
    ) -> None:
        if not legal_moves:
            raise ValueError(
                f"{self.name} received no legal moves."
            )

    def __repr__(self) -> str:
        return (
            f"{self.__class__.__name__}("
            f"name={self.name!r}, "
            f"policy_type={self.policy_type!r})"
        )


class RandomPolicy(BasePolicy):

    def choose_move(
        self,
        state: Mapping[str, Any],
        legal_moves: Sequence[Any],
    ) -> AgentDecision:

        self.validate_legal_moves(legal_moves)

        move = self._rng.choice(
            list(legal_moves)
        )

        return AgentDecision(
            move=move,
            reason="Random legal move",
            metadata={"policy": self.name},
        )


class GreedyPolicy(BasePolicy):

    def choose_move(
        self,
        state: Mapping[str, Any],
        legal_moves: Sequence[Any],
    ) -> AgentDecision:

        self.validate_legal_moves(legal_moves)

        scores = state.get(
            "move_scores",
            {},
        )

        best_move = max(
            legal_moves,
            key=lambda move: scores.get(move, 0.0),
        )

        return AgentDecision(
            move=best_move,
            score=float(scores.get(best_move, 0.0)),
            reason="Highest immediate value",
            metadata={"policy": self.name},
        )


class HeuristicPolicy(BasePolicy):

    def choose_move(
        self,
        state: Mapping[str, Any],
        legal_moves: Sequence[Any],
    ) -> AgentDecision:

        self.validate_legal_moves(legal_moves)

        scores = state.get(
            "heuristic_scores",
            {},
        )

        best_move = max(
            legal_moves,
            key=lambda move: scores.get(
                move,
                float("-inf"),
            ),
        )

        return AgentDecision(
            move=best_move,
            score=float(scores.get(best_move, 0.0)),
            reason="Highest heuristic evaluation",
            metadata={"policy": self.name},
        )


class SearchPolicy(BasePolicy):

    def choose_move(
        self,
        state: Mapping[str, Any],
        legal_moves: Sequence[Any],
    ) -> AgentDecision:

        self.validate_legal_moves(legal_moves)

        scores = state.get(
            "search_scores",
            {},
        )

        best_move = max(
            legal_moves,
            key=lambda move: scores.get(
                move,
                float("-inf"),
            ),
        )

        return AgentDecision(
            move=best_move,
            score=float(scores.get(best_move, 0.0)),
            reason="Search engine recommendation",
            metadata={
                "policy": self.name,
                "search_depth": self.config.search_depth,
            },
        )
'''

policies_file = AGENTS_DIR / "policies.py"

policies_file.write_text(
    policies_module,
    encoding="utf-8",
)

print("[OK]", policies_file)

[OK] D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\agents\policies.py


# Cell 29 — Export base_agent.py and factories.py

In [32]:
base_agent_module = '''from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Mapping, Sequence

from .decisions import AgentDecision
from .policies import BasePolicy


@dataclass(slots=True)
class BaseAgent:
    name: str
    policy: BasePolicy

    metadata: dict[str, Any] = field(
        default_factory=dict
    )

    def choose_move(
        self,
        state: Mapping[str, Any],
        legal_moves: Sequence[Any],
    ) -> AgentDecision:

        return self.policy.choose_move(
            state=state,
            legal_moves=legal_moves,
        )

    def __repr__(self) -> str:
        return (
            f"{self.__class__.__name__}("
            f"name={self.name!r}, "
            f"policy={self.policy.name!r})"
        )
'''

factories_module = '''from __future__ import annotations

from .base_agent import BaseAgent
from .configs import PolicyConfig
from .policies import (
    BasePolicy,
    GreedyPolicy,
    HeuristicPolicy,
    RandomPolicy,
    SearchPolicy,
)


class PolicyFactory:

    _registry = {
        "random": RandomPolicy,
        "greedy": GreedyPolicy,
        "heuristic": HeuristicPolicy,
        "search": SearchPolicy,
    }

    @classmethod
    def create(
        cls,
        config: PolicyConfig,
    ) -> BasePolicy:

        try:
            policy_class = cls._registry[
                config.policy_type
            ]
        except KeyError as error:
            raise ValueError(
                f"Unknown policy "
                f"{config.policy_type!r}"
            ) from error

        return policy_class(config)

    @classmethod
    def available_policies(
        cls,
    ) -> list[str]:
        return sorted(cls._registry)


class AgentFactory:

    @staticmethod
    def create(
        name: str,
        policy_type: str,
        **kwargs,
    ) -> BaseAgent:

        config = PolicyConfig(
            name=f"{policy_type.title()}Policy",
            policy_type=policy_type,
            **kwargs,
        )

        return BaseAgent(
            name=name,
            policy=PolicyFactory.create(config),
        )
'''

base_agent_file = AGENTS_DIR / "base_agent.py"
factories_file = AGENTS_DIR / "factories.py"

base_agent_file.write_text(
    base_agent_module,
    encoding="utf-8",
)

factories_file.write_text(
    factories_module,
    encoding="utf-8",
)

print("[OK]", base_agent_file)
print("[OK]", factories_file)

[OK] D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\agents\base_agent.py
[OK] D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\agents\factories.py


# Cell 30 — Export __init__.py and Validate

In [33]:
init_module = '''from .base_agent import BaseAgent
from .configs import PolicyConfig
from .decisions import AgentDecision
from .factories import AgentFactory, PolicyFactory
from .policies import (
    BasePolicy,
    GreedyPolicy,
    HeuristicPolicy,
    RandomPolicy,
    SearchPolicy,
)

__all__ = [
    "AgentDecision",
    "PolicyConfig",
    "BasePolicy",
    "RandomPolicy",
    "GreedyPolicy",
    "HeuristicPolicy",
    "SearchPolicy",
    "BaseAgent",
    "PolicyFactory",
    "AgentFactory",
]
'''

init_file = AGENTS_DIR / "__init__.py"

init_file.write_text(
    init_module,
    encoding="utf-8",
)

expected_agent_files = [
    init_file,
    decisions_file,
    configs_file,
    policies_file,
    base_agent_file,
    factories_file,
]

for path in expected_agent_files:
    assert path.exists(), path
    print(
        f"[OK] {path.name:<20} "
        f"{path.stat().st_size:>6,} bytes"
    )

print()
print("src/agents package exported successfully.")

[OK] __init__.py             512 bytes
[OK] decisions.py            304 bytes
[OK] configs.py              494 bytes
[OK] policies.py           3,703 bytes
[OK] base_agent.py           802 bytes
[OK] factories.py          1,390 bytes

src/agents package exported successfully.
